# Problem / Incident Generator

Two completely separate pipelines:

## Problem → `data/questions/<slug>/`
- `Question.md`, `metadata.json`
- `testcase_generator.py` → produces `testcases.txt`, `testcases_submission.txt`
- `drivers/<lang>/signature.<ext>` + `driver.<ext>`
- LLM solves → `solution.py` → execute → `output.txt`

## Incident → `data/incidents/<slug>/`
- `IncidentReport.md`, `manifest.json`, `metadata.json`
- `language/<lang>/src/service.py` (buggy)
- `language/<lang>/tests/test_basic.py`, `test_edgecases.py`
- `language/<lang>/run_tests.sh`
- Manifest embeds all file contents

In [ ]:
from pathlib import Path
import json
from typing import Optional

ROOT = Path.cwd()
DATA_ROOT = ROOT / "data"
QUESTIONS_ROOT = DATA_ROOT / "questions"
INCIDENTS_ROOT = DATA_ROOT / "incidents"

OUTPUT_MODE = "incident"  # "problem" or "incident"
IDEA = "Design a GPU Inference Scheduler"
LANGS = ["java", "python", "C++"]
ADDITIONAL_INFO_ON_IDEA = """
Incoming requests

Prompt

Priority

Tokens

Deadline

Need scheduler

Policies

FIFO

Priority

SJF

Token-aware

Fair Share

Need

Batching

Cancellation

Queue timeout

Backpressure

"""
SLUG_NAME = "GPU Inference Scheduler"
REFERENCE_SLUG = ""
MODEL = "openai/openai/gpt-oss-20b"
API_BASE = "http://localhost:1234/v1"
TEMPERATURE = 0

WRITE_OUTPUTS = True
RUN_LOCAL_VALIDATION = True

print("OUTPUT_MODE:", OUTPUT_MODE)
print("IDEA:", IDEA)
print("LANGS:", LANGS)


OUTPUT_MODE: incident
IDEA: Design a GPU Inference Scheduler
LANGS: ['java', 'python', 'C++']


In [42]:
import re
import subprocess
import tempfile
import litellm
from dataclasses import dataclass

litellm.api_base = API_BASE
litellm.api_key = "lm-studio"

def llm_complete(prompt: str) -> str:
    resp = litellm.completion(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=TEMPERATURE,
    )
    return resp.choices[0].message.content.strip()

def strip_code_fences(text: str) -> str:
    text = text.strip()
    text = re.sub(r'^```[a-zA-Z0-9_+\-]*\n', '', text)
    text = re.sub(r'\n```$', '', text)
    return text.strip()


In [ ]:
def to_slug(text: str) -> str:
    cleaned = re.sub(r"[^a-zA-Z0-9]+", "-", text.strip().lower())
    cleaned = re.sub(r"-+", "-", cleaned).strip("-")
    return cleaned or "untitled"

@dataclass
class Package:
    mode: str
    slug: str
    root: Path

def package_root(mode: str, slug: str) -> Path:
    if mode == "problem":
        return QUESTIONS_ROOT / slug
    return INCIDENTS_ROOT / slug

def read_small(path: Path, limit: int = 3000) -> str:
    if not path.exists():
        return ""
    return path.read_text(encoding="utf-8")[:limit]

def find_reference() -> Optional[Path]:
    if REFERENCE_SLUG:
        for base in (QUESTIONS_ROOT, INCIDENTS_ROOT):
            candidate = base / to_slug(REFERENCE_SLUG)
            if candidate.exists():
                return candidate
    if OUTPUT_MODE == "problem":
        search_roots = [QUESTIONS_ROOT, INCIDENTS_ROOT]
    else:
        search_roots = [INCIDENTS_ROOT, QUESTIONS_ROOT]
    for base in search_roots:
        if base.exists():
            items = sorted([p for p in base.iterdir() if p.is_dir()])
            if items:
                return items[0]
    return None

slug = to_slug(SLUG_NAME or IDEA)
pkg = Package(mode=OUTPUT_MODE, slug=slug, root=package_root(OUTPUT_MODE, slug))
pkg.root.mkdir(parents=True, exist_ok=True)

reference_dir = find_reference()
reference_text = ""
if reference_dir:
    if (reference_dir / "Question.md").exists():
        reference_text = read_small(reference_dir / "Question.md")
    elif (reference_dir / "IncidentReport.md").exists():
        reference_text = read_small(reference_dir / "IncidentReport.md")
    elif (reference_dir / "manifest.json").exists():
        reference_text = read_small(reference_dir / "manifest.json")

print("package root:", pkg.root)
print("output mode:", OUTPUT_MODE)
print("reference:", reference_dir)


TypeError: unsupported operand type(s) for |: 'type' and 'NoneType'

In [ ]:
full_idea = IDEA
if ADDITIONAL_INFO_ON_IDEA:
    full_idea += f"\n\nAdditional context: {ADDITIONAL_INFO_ON_IDEA}"

reference_guidance = ""
if reference_text:
    reference_guidance = f"\n\nReference style guidance (use only as style reference):\n{reference_text[:2000]}"

PLAN_PROMPT = f"""
You are designing a small repo package.

Idea: {full_idea}
Mode: {OUTPUT_MODE}{reference_guidance}

Return a concise implementation plan as markdown with these sections:
- package goal
- artifact list
- testcase strategy
- language-wise generation plan
- validation plan

Keep it practical and small.
""".strip()

if OUTPUT_MODE == "problem":
    QUESTION_PROMPT = f"""
    Write only the full markdown for a new algorithm problem.

    Idea: {full_idea}{reference_guidance}

    The output must include:
    - title
    - context
    - problem statement
    - examples
    - constraints
    - hints
    - no solution
    """.strip()
else:
    QUESTION_PROMPT = f"""
    Write only the full incident report markdown for a new codebase bug.

    Idea: {full_idea}{reference_guidance}

    The output must include:
    - challenge type
    - difficulty
    - context
    - user task
    - constraints
    - no fix
    """.strip()

print('prompts ready')


In [ ]:
plan_md = llm_complete(PLAN_PROMPT)
plan_path = pkg.root / "plan.md"
if WRITE_OUTPUTS:
    plan_path.write_text(plan_md + "\n", encoding="utf-8")
print(plan_md[:1200])


In [ ]:
description_md = llm_complete(QUESTION_PROMPT)
if OUTPUT_MODE == "problem":
    desc_path = pkg.root / "Question.md"
else:
    desc_path = pkg.root / "IncidentReport.md"
if WRITE_OUTPUTS:
    desc_path.write_text(description_md + "\n", encoding="utf-8")
print(description_md[:1200])


In [ ]:
if OUTPUT_MODE == "problem":
    testcase_prompt = f'''You are writing a deterministic Python script named testcase_generator.py.

The script must:
- generate testcases.txt and testcases_submission.txt
- contain a pure oracle solve function in Python
- compute expected outputs deterministically
- not use an LLM inside the script
- keep cases small for local tests and larger for submission tests

Question description:
{description_md[:3500]}

Return only the Python script.
'''.strip()
    testcase_script = llm_complete(testcase_prompt)
    testcase_script = strip_code_fences(testcase_script)
    testcase_path = pkg.root / "testcase_generator.py"
    if WRITE_OUTPUTS:
        testcase_path.write_text(testcase_script + "\n", encoding="utf-8")
    print(testcase_script[:1200])
else:
    print("Skipping testcase generator for incident mode.")


In [ ]:
if OUTPUT_MODE == "problem":
    LANG_EXT = {"java": "java", "python": "py", "C++": "cpp"}
    generated_signature_paths = {}
    generated_driver_paths = {}

    for lang in LANGS:
        lang_dir = pkg.root / "drivers" / lang
        lang_dir.mkdir(parents=True, exist_ok=True)

        sig_prompt = f"""Generate only the function signature/stub for this package.
Language: {lang}
Mode: {OUTPUT_MODE}

Question summary:
{description_md[:1500]}

Output code only.
""".strip()

        drv_prompt = f"""Generate only the language driver for this package.
Language: {lang}
Mode: {OUTPUT_MODE}

The driver must:
- read testcases.txt
- call the user solution
- write output.json with results

Question summary:
{description_md[:1500]}

Output code only.
""".strip()

        signature_code = strip_code_fences(llm_complete(sig_prompt))
        driver_code = strip_code_fences(llm_complete(drv_prompt))

        if lang not in LANG_EXT:
            raise ValueError(f"Unsupported language: {lang}")
        sig_path = lang_dir / f"signature.{LANG_EXT[lang]}"
        drv_path = lang_dir / f"driver.{LANG_EXT[lang]}"
        if WRITE_OUTPUTS:
            sig_path.write_text(signature_code + "\n", encoding="utf-8")
            drv_path.write_text(driver_code + "\n", encoding="utf-8")

        generated_signature_paths[lang] = sig_path
        generated_driver_paths[lang] = drv_path

    print(generated_signature_paths)
    print(generated_driver_paths)
else:
    print("Skipping drivers/signatures for incident mode.")


In [ ]:
if OUTPUT_MODE == "problem":
    manifest = {
        "slug": pkg.slug,
        "mode": OUTPUT_MODE,
        "idea": IDEA,
        "title": description_md.splitlines()[0].replace("# ", "") if description_md.startswith("#") else IDEA,
        "difficulty": "Medium",
        "tags": ["Generated", "LLM"],
        "tests": ["testcases.txt", "testcases_submission.txt"],
        "drivers": {lang: f"drivers/{lang}" for lang in LANGS},
        "solution": "solution.py",
        "output": "output.txt",
    }
else:
    manifest = {
        "version": 1,
        "report": description_md,
        "availableLanguages": LANGS,
        "defaultLanguage": LANGS[0] if LANGS else "python",
        "entryFileByLanguage": {},
        "filesByLanguage": {},
    }

manifest_path = pkg.root / "manifest.json"
if WRITE_OUTPUTS:
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
print(manifest)


In [ ]:
if OUTPUT_MODE == "incident":
    for lang in LANGS:
        lang_prompt = f"""Generate the file contents for a {OUTPUT_MODE} package in {lang}.

Idea: {full_idea}
Incident description:
{description_md[:3000]}

Return a JSON object with exactly these keys (no markdown, no code fences):
{{
  "service": "<full contents of src/service.py>",
  "test_basic": "<full contents of tests/test_basic.py>",
  "test_edgecases": "<full contents of tests/test_edgecases.py>",
  "run_tests": "<full contents of run_tests.sh>"
}}
""".strip()

        raw = llm_complete(lang_prompt)
        raw = strip_code_fences(raw)

        try:
            files = json.loads(raw)
        except json.JSONDecodeError:
            raise ValueError(f"LLM did not return valid JSON for {lang}: {raw[:500]}")

        lang_root = pkg.root / "language" / lang
        lang_root.mkdir(parents=True, exist_ok=True)
        (lang_root / "src").mkdir(exist_ok=True)
        (lang_root / "tests").mkdir(exist_ok=True)

        (lang_root / "src" / "service.py").write_text(files["service"] + "\n", encoding="utf-8")
        (lang_root / "tests" / "test_basic.py").write_text(files["test_basic"] + "\n", encoding="utf-8")
        (lang_root / "tests" / "test_edgecases.py").write_text(files["test_edgecases"] + "\n", encoding="utf-8")
        (lang_root / "run_tests.sh").write_text(files["run_tests"] + "\n", encoding="utf-8")

        if "filesByLanguage" not in manifest:
            manifest["filesByLanguage"] = {}
        if "entryFileByLanguage" not in manifest:
            manifest["entryFileByLanguage"] = {}

        manifest["filesByLanguage"][lang] = [
            {"path": "src/service.py", "content": files["service"], "readonly": False, "language": lang},
            {"path": "tests/test_basic.py", "content": files["test_basic"], "readonly": True, "language": lang},
            {"path": "tests/test_edgecases.py", "content": files["test_edgecases"], "readonly": True, "language": lang},
        ]
        manifest["entryFileByLanguage"][lang] = "src/service.py"

    if WRITE_OUTPUTS:
        manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    print("generated incident files for", LANGS)
else:
    print("Skipping incident file generation.")


In [ ]:
if OUTPUT_MODE == "problem":
    def local_problem_solve(question_text: str) -> str:
        prompt = f"""Solve the following problem as a concise Python function implementation.
Return only code.

{question_text[:5000]}
""".strip()
        return strip_code_fences(llm_complete(prompt))

    def execute_python_solution(solution_code: str, testcase_file: Path, driver_file: Path) -> str:
        with tempfile.TemporaryDirectory() as td:
            td = Path(td)
            (td / "solution.py").write_text(solution_code + "\n", encoding="utf-8")
            (td / "driver.py").write_text(driver_file.read_text(encoding="utf-8"), encoding="utf-8")
            (td / "testcases.txt").write_text(testcase_file.read_text(encoding="utf-8"), encoding="utf-8")
            proc = subprocess.run(["python3", "driver.py"], cwd=td, capture_output=True, text=True)
            if proc.returncode != 0:
                raise RuntimeError(proc.stderr)
            return proc.stdout

    solution_code = local_problem_solve(description_md)
    solution_path = pkg.root / "solution.py"
    if WRITE_OUTPUTS:
        solution_path.write_text(solution_code + "\n", encoding="utf-8")

    output_text = execute_python_solution(solution_code, testcase_path, generated_driver_paths["python"])
    output_path = pkg.root / "output.txt"
    if WRITE_OUTPUTS:
        output_path.write_text(output_text, encoding="utf-8")
    print(output_text[:2000])
else:
    print("Skipping solve/output generation for incident mode.")


In [ ]:
if OUTPUT_MODE == "problem":
    title = description_md.splitlines()[0].replace("# ", "") if description_md.startswith("#") else IDEA
    metadata = {
        "slug": pkg.slug,
        "title": title,
        "difficulty": "Medium",
        "tags": ["Generated", "LLM"],
        "hints": ["Keep the testcase generator deterministic.", "Use a compact plan before writing artifacts."]
    }
else:
    title_line = description_md.splitlines()[0] if description_md else IDEA
    metadata = {
        "slug": pkg.slug,
        "title": title_line,
        "severity": "P0",
        "difficulty": "Medium",
        "service": "Generated Service",
        "summary": IDEA,
        "slaMinutes": 20
    }

metadata_path = pkg.root / "metadata.json"
if WRITE_OUTPUTS:
    metadata_path.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
print(metadata)


In [ ]:
def validate_problem(pkg: Package) -> None:
    assert (pkg.root / "Question.md").exists()
    assert (pkg.root / "testcase_generator.py").exists()
    assert (pkg.root / "manifest.json").exists()
    assert (pkg.root / "output.txt").exists()

def validate_incident(pkg: Package) -> None:
    assert (pkg.root / "IncidentReport.md").exists()
    assert (pkg.root / "manifest.json").exists()
    for lang in LANGS:
        lang_root = pkg.root / "language" / lang
        assert (lang_root / "src" / "service.py").exists()
        assert (lang_root / "tests" / "test_basic.py").exists()
        assert (lang_root / "tests" / "test_edgecases.py").exists()
        assert (lang_root / "run_tests.sh").exists()

if RUN_LOCAL_VALIDATION:
    if OUTPUT_MODE == "problem":
        validate_problem(pkg)
    else:
        validate_incident(pkg)
    print("validation passed")

print("generated at:", pkg.root.resolve())
